<a href="https://colab.research.google.com/github/yemanegm1101/Ansys-MAPDL-Practice-Problems/blob/main/studio_tripo3d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Wed Sep 16 12:08:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%cd /content
!git clone https://github.com/VAST-AI-Research/TripoSG.git
%cd /content/TripoSG
!ls

/content
fatal: destination path 'TripoSG' already exists and is not an empty directory.
/content/TripoSG
assets	LICENSE  NOTICE  README.md  requirements.txt  scripts  triposg


In [3]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
    print("VRAM:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA: 12.8
VRAM: 14.56 GB


In [12]:
import os

!pip install -U pip wheel

# Force install setuptools to a version compatible with torch before other packages.
# Torch requires setuptools<82, so we'll install 81.0.0.
!pip install setuptools==81.0.0

%cd /content/TripoSG

# Remove the problematic numpy version constraint from requirements.txt
!sed -i '/numpy==1.22.3/d' requirements.txt

# Re-include 'diso' to attempt installation, as it is a required dependency.
# !sed -i '/diso/d' requirements.txt

!pip install -r requirements.txt

/content/TripoSG


In [1]:
import os

%cd /content/TripoSG

# Install build dependencies for native extensions
!apt-get update
!apt-get install -y build-essential python3-dev

!pip install diso --verbose --no-build-isolation

/content/TripoSG
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:4 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Reading package lists... Done
W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source

In [2]:
import torch
import trimesh
import transformers
import diffusers

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("Trimesh:", trimesh.__version__)
print("Transformers:", transformers.__version__)
print("Diffusers:", diffusers.__version__)

Torch: 2.11.0+cu128
CUDA: True
Trimesh: 5.1.0
Transformers: 5.16.1
Diffusers: 0.40.0


In [3]:
!python -c "import triposg; print('TripoSG import: OK')"

TripoSG import: OK


In [4]:
%cd /content/TripoSG

!python -m scripts.inference_triposg \
    --image-input assets/example_data/hjswed.png \
    --output-path ./output.glb

/content/TripoSG
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0% 0/11 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/436 [00:00<?, ?B/s]           

Fetching 11 files:   9% 1/11 [00:00<00:01,  7.71it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):  28% 436/1.54k [00:00<00:00, 3.39kB/s]
Reconstructing (incomplete total...):  51% 1.54k/3.00k [00:00<00:00, 3.39kB/s]
Reconstructing (incomplete total...):   0% 3.00k/1.22G [00:00<99:41:10, 3.39kB/s]
Reconstructing (incomplete total...):   0% 3.00k/6.98G [00:00<571:09:07, 3.39kB/s]
Reconstructing (incomplete total...):   0% 3.00k/6.98G [00:00<571:09:07, 3.39kB/s]
Reconstructing (incomplete total...):   0% 3.53k/7.95G [00:00<650:37:41, 3.39kB/s]
Reconstructing (incomplete total...):   0% 3.53k/7.95G [00:00<650:37:41, 3.39kB/s]
Reconstructing 

In [6]:
import os

print(os.path.exists("/content/TripoSG/output.glb"))

if os.path.exists("/content/TripoSG/output.glb"):
    print("Output size:",
          round(os.path.getsize("/content/TripoSG/output.glb") / 1024**2, 2),
          "MB")

True
Output size: 22.58 MB


In [8]:
from google.colab import files

files.download("/content/TripoSG/output.glb")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
from google.colab import files
import os
import subprocess
from IPython.display import display, HTML

# Ensure we are in the TripoSG directory
os.chdir("/content/TripoSG")

# Allow the user to upload a file
print("Please upload your image file:")
uploaded = files.upload()

# Get the name of the uploaded file
if not uploaded:
    print("No file was uploaded. Please upload an image to proceed.")
    raise ValueError("No file uploaded.")

uploaded_filename = next(iter(uploaded.keys()))

# Since os.chdir("/content/TripoSG") was executed, files.upload() places
# the file directly into /content/TripoSG/.
# We want to rename this file to my_object.png within the same directory.
current_working_dir = os.getcwd()
path_to_uploaded_file = os.path.join(current_working_dir, uploaded_filename)

target_image_filename = "my_object.png"
target_image_path = os.path.join(current_working_dir, target_image_filename)
output_glb_path = os.path.join(current_working_dir, "my_object.glb") # Standardize output name as well

# Rename the uploaded file to my_object.png inside /content/TripoSG/
print(f"Moving and renaming '{path_to_uploaded_file}' to '{target_image_path}'...")
!mv "{path_to_uploaded_file}" "{target_image_path}"
print(f"File is now available at: {target_image_path}")

print(f"Input image for inference: {target_image_path}")
print(f"Output GLB model will be saved to: {output_glb_path}")

command = [
    "python",
    "-m",
    "scripts.inference_triposg",
    "--image-input",
    target_image_path,
    "--faces",
    "10000",
    "--output-path",
    output_glb_path
]

try:
    print("\nStarting TripoSG inference...")
    # Execute the inference script
    result = subprocess.run(command, check=True, capture_output=True, text=True)
    print("Inference completed successfully.")
    print("Stdout:", result.stdout)
    if result.stderr:
        print("Stderr:", result.stderr)

    # Download the generated GLB file
    if os.path.exists(output_glb_path):
        print(f"\nDownloading generated model: {output_glb_path}")
        files.download(output_glb_path)

        # Embed the model-viewer web component for preview
        print(f"\nDisplaying generated model: {output_glb_path}")
        html_viewer = f"""
        <script type="module" src="https://ajax.googleapis.com/ajax/libs/model-viewer/3.4.0/model-viewer.min.js"></script>
        <model-viewer
            src="file://{output_glb_path}"
            alt="A 3D model of your object"
            ar
            ar-modes="webxr scene-viewer quick-look"
            shadow-intensity="1"
            camera-controls
            touch-action="pan-y"
            style="width: 600px; height: 400px;"
        ></model-viewer>
        """
        display(HTML(html_viewer))
    else:
        print(f"Error: Output file {output_glb_path} not found after inference.")

except subprocess.CalledProcessError as e:
    print(f"\nError during TripoSG inference: {e}")
    print(f"Command failed with exit code {e.returncode}")
    if e.stdout:
        print(f"Stdout:\n{e.stdout}")
    if e.stderr:
        print(f"Stderr:\n{e.stderr}")
    print("Please check the error messages above for details. It's possible the input image or script parameters caused an issue.")
    raise # Re-raise the exception to indicate failure to the user.
except Exception as e:
    print(f"\nAn unexpected error occurred: {e}")
    raise

Please upload your image file:


Saving Modern Metal Harp Fountain Musical Water Installation.jpg to Modern Metal Harp Fountain Musical Water Installation (3).jpg
Moving and renaming '/content/TripoSG/Modern Metal Harp Fountain Musical Water Installation (3).jpg' to '/content/TripoSG/my_object.png'...
File is now available at: /content/TripoSG/my_object.png
Input image for inference: /content/TripoSG/my_object.png
Output GLB model will be saved to: /content/TripoSG/my_object.glb

Starting TripoSG inference...
Inference completed successfully.
Stdout: Loading weights from local directory
final grids shape =  torch.Size([505, 505, 505])
Mesh saved to /content/TripoSG/my_object.glb

Stderr: 
Fetching 11 files: 100%|██████████| 11/11 [00:00<00:00, 465.52it/s]

Fetching 20 files: 100%|██████████| 20/20 [00:00<00:00, 550.69it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

Loading weights:  19%|█▉        | 83/439 [00:00<00:00, 822.90it/s]



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Displaying generated model: /content/TripoSG/my_object.glb


In [27]:
import trimesh
from google.colab import files
import os

# Define the path to the GLB model
model_path = "/content/TripoSG/my_object.glb"

# Check if the GLB model exists
if not os.path.exists(model_path):
    print(f"Error: GLB model not found at {model_path}")
else:
    # Load the GLB model
    print(f"Loading GLB model from: {model_path}")
    try:
        mesh = trimesh.load(model_path)
        print("GLB model loaded successfully.")

        # Define output paths for OBJ and STL
        obj_output_path = "/content/TripoSG/my_object.obj"
        stl_output_path = "/content/TripoSG/my_object.stl"

        # Export to OBJ
        print(f"Exporting to OBJ: {obj_output_path}")
        mesh.export(obj_output_path)
        print("OBJ conversion complete.")

        # Export to STL
        print(f"Exporting to STL: {stl_output_path}")
        mesh.export(stl_output_path)
        print("STL conversion complete.")

        # Offer to download the converted files
        if os.path.exists(obj_output_path):
            print(f"\nDownloading {obj_output_path}")
            files.download(obj_output_path)
        if os.path.exists(stl_output_path):
            print(f"Downloading {stl_output_path}")
            files.download(stl_output_path)

    except Exception as e:
        print(f"An error occurred during model conversion: {e}")

Loading GLB model from: /content/TripoSG/my_object.glb
GLB model loaded successfully.
Exporting to OBJ: /content/TripoSG/my_object.obj
OBJ conversion complete.
Exporting to STL: /content/TripoSG/my_object.stl
STL conversion complete.



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Optimize Model: Reduce Polygon Count

To optimize the model for performance, especially for web or real-time applications, you can reduce its polygon count (mesh decimation). The `trimesh` library provides functions to do this while trying to preserve the model's visual integrity.

In [ ]:
import trimesh
from google.colab import files
import os

# Path to the original GLB model
model_path = "/content/TripoSG/my_object.glb"

# Check if the GLB model exists
if not os.path.exists(model_path):
    print(f"Error: GLB model not found at {model_path}")
else:
    # Load the GLB model
    print(f"Loading original GLB model from: {model_path}")
    try:
        mesh = trimesh.load(model_path)
        print(f"Original mesh has {len(mesh.faces)} faces.")

        # Define target face count (e.g., 20% of original, or a fixed number)
        # You can adjust this value based on your optimization needs.
        target_faces = int(len(mesh.faces) * 0.2)
        if target_faces < 1000: # Ensure a minimum for reasonable detail
            target_faces = 1000
        print(f"Attempting to reduce faces to approximately {target_faces}.")

        # Perform quadric decimation to simplify the mesh
        simplified_mesh = mesh.simplify_quadric_decimation(target_faces=target_faces)

        print(f"Simplified mesh has {len(simplified_mesh.faces)} faces.")

        # Define output path for the optimized GLB model
        optimized_output_path = "/content/TripoSG/my_object_optimized.glb"

        # Export the simplified model
        print(f"Exporting optimized GLB to: {optimized_output_path}")
        simplified_mesh.export(optimized_output_path)
        print("Optimized GLB conversion complete.")

        # Offer to download the optimized file
        if os.path.exists(optimized_output_path):
            print(f"\nDownloading optimized model: {optimized_output_path}")
            files.download(optimized_output_path)

    except Exception as e:
        print(f"An error occurred during model optimization: {e}")

In [ ]:
from IPython.display import display, HTML

# Path to your generated GLB model
model_path = "/content/TripoSG/my_object.glb"

# Check if the model exists before trying to display it
if os.path.exists(model_path):
    # Embed the model-viewer web component
    html_viewer = f"""
    <script type="module" src="https://ajax.googleapis.com/ajax/libs/model-viewer/3.4.0/model-viewer.min.js"></script>
    <model-viewer
        src="file://{model_path}"
        alt="A 3D model of your object"
        ar
        ar-modes="webxr scene-viewer quick-look"
        shadow-intensity="1"
        camera-controls
        touch-action="pan-y"
        style="width: 600px; height: 400px;"
    ></model-viewer>
    """
    display(HTML(html_viewer))
else:
    print(f"Error: Model not found at {model_path}")
    print("Please ensure the inference process completed successfully and the .glb file was generated.")

In [25]:
from IPython.display import display, HTML

# Path to your generated GLB model
model_path = "/content/TripoSG/my_object.glb"

# Check if the model exists before trying to display it
if os.path.exists(model_path):
    # Embed the model-viewer web component
    html_viewer = f"""
    <script type="module" src="https://ajax.googleapis.com/ajax/libs/model-viewer/3.4.0/model-viewer.min.js"></script>
    <model-viewer
        src="file://{model_path}"
        alt="A 3D model of your object"
        ar
        ar-modes="webxr scene-viewer quick-look"
        shadow-intensity="1"
        camera-controls
        touch-action="pan-y"
        style="width: 600px; height: 400px;"
    ></model-viewer>
    """
    display(HTML(html_viewer))
else:
    print(f"Error: Model not found at {model_path}")
    print("Please ensure the inference process completed successfully and the .glb file was generated.")